# Lecture 13 - Iterators, Iterables, and the itertools Module

## Learning Objectives

- Understand the iterator protocol: iter() and next()
- Write generators with yield for memory-efficient streams
- Chain, cycle, and count with itertools
- Group data with itertools.groupby
- Generate combinations, permutations, and products

## Key Topics

- iter() and next()
- Custom iterators vs generators (yield)
- itertools.chain, itertools.cycle, itertools.count
- itertools.groupby for grouped operations
- itertools.product, itertools.combinations, itertools.permutations

## `iter()` and `next()`

An **iterable** is any Python object that can return its elements one at a time — lists, tuples, strings, dictionaries, files, etc. An **iterator** is an object that remembers its position during iteration.

You obtain an iterator from an iterable using `iter()`, and you step through its values using `next()`. When the iterator is exhausted, `next()` raises `StopIteration`. This is exactly what happens behind the scenes when you write `for item in my_list`.

In [ ]:
# Manual iteration with iter() and next()
fruits = ["apple", "banana", "cherry"]
iterator = iter(fruits)

print(next(iterator))
print(next(iterator))
print(next(iterator))

# Uncommenting the next line would raise StopIteration:
# print(next(iterator))


In [ ]:
# Handling StopIteration gracefully
def manual_iterate(iterable):
    it = iter(iterable)
    while True:
        try:
            item = next(it)
            print(f"Got: {item}")
        except StopIteration:
            print("Iterator exhausted.")
            break

manual_iterate([10, 20, 30])


## Custom Iterators vs Generators (`yield`)

An **iterator** is a class that implements `__iter__()` (returns self) and `__next__()` (returns the next item). A **generator** is a simpler way to create iterators using the `yield` keyword in a function. Each time `yield` is reached, the function's state is frozen, and the value is returned to the caller.

Generators are the most common way to write custom iterators in Python because they are less boilerplate than writing a full iterator class.

In [ ]:
# Generator function with yield
def countdown(n):
    while n > 0:
        yield n
        n -= 1

for num in countdown(5):
    print(num)

# Convert to list
print(list(countdown(5)))


In [ ]:
# Generator for streaming data (simulated)
def read_sensor_data(num_readings):
    """Simulate streaming sensor readings."""
    import random
    for i in range(num_readings):
        yield {
            "reading_id": i + 1,
            "temperature": round(random.uniform(20.0, 30.0), 2),
            "humidity": round(random.uniform(40.0, 70.0), 2),
        }

for reading in read_sensor_data(5):
    print(f"ID {reading['reading_id']}: {reading['temperature']}C, {reading['humidity']}%")


## `itertools.chain`, `itertools.cycle`, `itertools.count`

The `itertools` module is a collection of tools for building efficient iterators:

- **`chain`**: Combine multiple iterables into one sequential iterator.
- **`cycle`**: Cycle through an iterable infinitely.
- **`count`**: Count from a start value infinitely with a configurable step.

These tools help you avoid creating intermediate lists and keep your code memory-efficient.

In [ ]:
import itertools

# chain — concatenate iterables
numbers = [1, 2, 3]
letters = ["a", "b", "c"]
combined = list(itertools.chain(numbers, letters))
print(f"Chained: {combined}")

# count — infinite counter (use with break to avoid infinite loop)
for i, val in enumerate(itertools.count(10, 5)):
    if i >= 5:
        break
    print(f"Count: {val}")

# cycle — repeat indefinitely (use with break)
colors = ["red", "green", "blue"]
cyclic = itertools.cycle(colors)
for i, color in enumerate(cyclic):
    if i >= 7:
        break
    print(f"Cycle {i}: {color}")


## `itertools.groupby` for Grouped Operations

`groupby` groups consecutive elements in an iterable that share a common key. It returns an iterator of `(key, group)` pairs, where `group` is itself an iterator of items. **Important**: `groupby` only groups consecutive matching items, so you often need to sort by the key first.

This is useful for aggregating data that arrives in sorted order, like log entries or time-series data.

In [ ]:
import itertools

# Simple grouping
data = ["apple", "apple", "banana", "banana", "banana", "cherry", "cherry"]
for key, group in itertools.groupby(data):
    print(f"Key: {key}, Count: {len(list(group))}")

print("---")

# Grouping records (must sort first for meaningful groups)
records = [
    {"city": "NYC", "sales": 100},
    {"city": "LA", "sales": 150},
    {"city": "NYC", "sales": 200},
    {"city": "CHI", "sales": 120},
    {"city": "LA", "sales": 180},
]
records.sort(key=lambda r: r["city"])
for city, group in itertools.groupby(records, key=lambda r: r["city"]):
    sales = [r["sales"] for r in group]
    print(f"{city}: total sales = {sum(sales)}, count = {len(sales)}")


## `product`, `combinations`, `permutations`

These itertools functions are essential for combinatorial operations:

- **`product`**: Cartesian product of input iterables (nested loop equivalent).
- **`combinations`**: All unique r-length tuples **without** regard to order.
- **`permutations`**: All unique r-length tuples **with** regard to order.

In data science, `combinations` is frequently used to generate pairs of features for interaction terms, and `product` is used for hyperparameter grid search.

In [ ]:
import itertools

# product — Cartesian product (grid search)
param_grid = {
    "learning_rate": [0.01, 0.1],
    "max_depth": [3, 5, 7],
    "n_estimators": [50, 100],
}
keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Grid search has {len(combos)} combinations")
for combo in combos[:3]:
    print(f"  {dict(zip(keys, combo))}")
print("  ...")


In [ ]:
# combinations — feature pairs for interaction terms
features = ["age", "income", "score", "education"]
pairs = list(itertools.combinations(features, 2))
print(f"Feature pairs for interactions: {pairs}")

# permutations — ordered sequences
letters = ["A", "B", "C"]
perms = list(itertools.permutations(letters, 2))
print(f"Permutations: {perms}")


## Data Science Connection

Iterators and generators are the engine behind streaming data pipelines. When you process a 10 GB CSV file, you cannot load it all into memory — generators let you process one row at a time. The `itertools` module provides high-performance building blocks for common data tasks: grouping records, generating feature combinations, and computing Cartesian products for hyperparameter grids.